# Silver Layer — Clean & Enrich Healthcare Data
Validate admissions, deduplicate records, derive LOS buckets and flag abnormal vitals.
Does NOT derive any readmission-based flags (avoids target leakage).

In [ ]:
from pyspark.sql.functions import (
    col, when, lit, to_timestamp, date_format, hour,
    row_number, current_timestamp
)
from pyspark.sql.window import Window

In [ ]:
# Pass-through staff catalog — already clean dimension
df_staff = spark.read.format('delta').table('bronze_staff_catalog')
df_staff = df_staff.withColumn('silver_timestamp', current_timestamp())
df_staff.write.mode('overwrite').format('delta').saveAsTable('silver_staff_catalog')
print(f'Silver staff catalog: {df_staff.count()} rows')

In [ ]:
# Clean patient admissions (no readmission-derived columns)
df_adm = spark.read.format('delta').table('bronze_patient_admissions')
w = Window.partitionBy('admission_id').orderBy(col('ingestion_timestamp').desc())
df_adm = (
    df_adm.withColumn('_rn', row_number().over(w)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('admission_date', to_timestamp('admission_date'))
    .withColumn('discharge_date', to_timestamp('discharge_date'))
    .withColumn('length_of_stay_days', col('length_of_stay_days').cast('int'))
    .withColumn('prior_admissions', col('prior_admissions').cast('int'))
    .withColumn('is_readmission', col('is_readmission').cast('boolean'))
    .filter(col('admission_date').isNotNull())
    .filter(col('length_of_stay_days') >= 0)
    .withColumn('los_bucket',
        when(col('length_of_stay_days') == 0, 'Same Day')
        .when(col('length_of_stay_days') <= 2, '1-2 Days')
        .when(col('length_of_stay_days') <= 7, '3-7 Days')
        .when(col('length_of_stay_days') <= 14, '8-14 Days')
        .otherwise('15+ Days'))
    .withColumn('dx_chapter', col('primary_dx_code').substr(1, 1))
    .withColumn('admission_date_only', date_format('admission_date', 'yyyy-MM-dd'))
    .withColumn('admission_shift',
        when(hour('admission_date') < 8, 'Night')
        .when(hour('admission_date') < 16, 'Day')
        .otherwise('Evening'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_adm.write.mode('overwrite').format('delta').saveAsTable('silver_patient_admissions')
print(f'Silver admissions: {df_adm.count()} rows')

In [ ]:
# Clean clinical records (vitals) + flag abnormal readings
df_clin = spark.read.format('delta').table('bronze_clinical_records')
w2 = Window.partitionBy('record_id').orderBy(col('ingestion_timestamp').desc())
df_clin = (
    df_clin.withColumn('_rn', row_number().over(w2)).filter(col('_rn') == 1).drop('_rn')
    .withColumn('recorded_at', to_timestamp('recorded_at'))
    .withColumn('value', col('value').cast('double'))
    .filter(col('recorded_at').isNotNull())
    .filter(col('value').isNotNull())
    .withColumn('is_abnormal',
        when((col('vital_type') == 'Blood Pressure Systolic') & ((col('value') < 90) | (col('value') > 140)), True)
        .when((col('vital_type') == 'Heart Rate') & ((col('value') < 60) | (col('value') > 100)), True)
        .when((col('vital_type') == 'Temperature') & ((col('value') < 36.1) | (col('value') > 37.8)), True)
        .when((col('vital_type') == 'O2 Saturation') & (col('value') < 95), True)
        .otherwise(False))
    .withColumn('recorded_date', date_format('recorded_at', 'yyyy-MM-dd'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_clin.write.mode('overwrite').format('delta').saveAsTable('silver_clinical_records')
print(f'Silver clinical records: {df_clin.count()} rows')